Quick script to load some `.npy` arrays and convert them into `.nii` so that they can be used for training SARDU-Net

In [16]:
# Handle imports
import numpy as np
import nibabel as nib

# Load the data
data = np.load("input_data/all_signals_all_substrates.npy")

# Inspect the shape of the data
print("Shape:", data.shape)
print("Data type:", data.dtype)


Shape: (4050, 16177)
Data type: float64


Reduce the number of time points in the series by only selecting certain $b$, $\Delta$, and $\delta$ values:

In [17]:
# Load the parameter lists
list_b = np.loadtxt("input_data/ref.bval")
list_delta = np.loadtxt("input_data/ref.gdur")
list_sep = np.loadtxt("input_data/ref.gsep")

# Create a boolean vector based on b values less/equal 2000, delta values equal to 4, and sep values less/equal 20
mask_b = (list_b <= 2000) & (list_delta == 4.0) & (list_sep <= 20.0) & (list_sep >= 8.0) 

# Extract sample of the data
data_sample = data[:, mask_b]

# Inspect the result
print("Original shape:", data.shape)
print("Selected shape:", data_sample.shape)
print("Number of selected parameter combinations:", np.sum(mask_b))

Original shape: (4050, 16177)
Selected shape: (4050, 126)
Number of selected parameter combinations: 126


Now, reshape the data into a 4D array that we can save as a NIFTI

In [18]:
data_4d = data_sample.reshape(25, 9, 18, data_sample.shape[1])

print("New shape:", data_4d.shape)


New shape: (25, 9, 18, 126)


Now we'll actually save it as a NIFTI

In [19]:
# Create a blank transformation matrix
affine = np.eye(4)

nifti = nib.Nifti1Image(data_4d, affine)

nib.save(nifti, "input_data/data_subsampled-126.nii")